#Project Setup

# AI Customer Support Agent

## Project Overview

This project implements an AI-powered Customer Support Agent using Retrieval-Augmented Generation (RAG).

The chatbot answers customer queries by retrieving information from a product knowledge base stored in Pinecone and generating responses using OpenAI models.

## Objectives

- Build an intelligent customer support chatbot
- Retrieve accurate product information using Pinecone
- Answer complex customer queries using RAG
- Recommend alternative products when requested items are unavailable
- Maintain conversation context
- Deploy the application using Streamlit

## Technology Stack

1.	Programming Language: Python 3.10
2.	Frameworks: LangChain, LangGraph, Streamlit
3.	LLM: OpenAI GPT-4o Mini
4.	Embedding Model: OpenAI text-embedding-3-small
5.	Vector Database: Pinecone Serverless
6.	Development Environment: Google Colab
7.	Knowledge Base: Jewellery Product Catalog, Warranty Policy, Shipping Policy, Return Policy, Jewellery Care Guide, FAQ


## Project Workflow

1. Install required libraries
2. Load API keys
3. Connect to Pinecone
4. Load product knowledge base
5. Retrieve relevant documents
6. Generate AI responses
7. Launch the Streamlit chatbot

## Flow Diagram

                          Jewellery PDF Documents
      (Product Catalog, Warranty, Shipping, Returns, Care Guide, FAQ)
                                      │
                                      ▼
                           Document Loading & Parsing
                                      │
                                      ▼
                   Semantic Chunking + Metadata Enhancement
                                      │
                                      ▼
                     OpenAI Embeddings (text-embedding-3-small)
                                      │
                                      ▼
                      Pinecone Vector Database (Serverless)
                                      │
                                      ▼
                              Customer Question
                                      │
                                      ▼
                             Initialize Agent
                                      │
                                      ▼
                        Intent Classification Agent
                                      │
                                      ▼
                           Query Rewriter Agent
                                      │
                                      ▼
                           Document Retrieval Agent
                                      │
                                      ▼
                             Response Agent (LLM)
                                      │
                                      ▼
                           Response Validation Agent
                                      │
                                      ▼
                          Recommendation Agent
                                      │
                                      ▼
                           Final Customer Response
                                      │
                                      ▼
                           Streamlit Chat Interface


## Features Implemented

1.	Multi-Agent Customer Support workflow using LangGraph.
2.	Retrieval-Augmented Generation (RAG) using a Pinecone vector database.
3.	Semantic document chunking with overlap to preserve context.
4.	Metadata enhancement for improved document retrieval accuracy.
5.	OpenAI embeddings for semantic similarity search.
6.	Intent Classification Agent to identify customer query categories.
7.	Query Rewriter Agent to transform follow-up questions into standalone queries for better retrieval.
8.	Intelligent document retrieval with metadata-based filtering and ranking.
9.	Response Generation Agent using retrieved knowledge only.
10.	Response Validation Agent to reduce hallucinations and ensure answers are grounded in retrieved documents.
11.	Recommendation Agent to suggest alternative products when an item is unavailable or when recommendations are requested.
12.	Conversation history support for multi-turn customer interactions.
13.	Source-aware responses generated only from the uploaded knowledge base.
14.	Automatic handling of unsupported or unavailable information with appropriate fallback responses.
15.	Modular agent architecture that separates retrieval, reasoning, validation, and recommendation tasks.
16.	Interactive Streamlit chat interface for real-time customer support.
17.	Cloud-based implementation using Google Colab, OpenAI, and Pinecone.


#Install Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install -q --upgrade openai streamlit langchain langchain-core langchain-community langchain-openai langchain-experimental langgraph pinecone crewai crewai-tools pymupdf pypdf tiktoken python-dotenv pandas numpy langchain-pinecone langchain-openai

### Note on Library Incompatibility:

If you encountered an `ImportError` related to `langchain_core` or `langchain_community` after running the `!pip install` command, it is likely due to a version mismatch that isn't fully resolved until the Python runtime is reset.

**To resolve this:**
1. Go to the Colab menu: **Runtime > Restart runtime...**
2. After the runtime restarts, **run all cells from the beginning** (Cell > Run all).

This will ensure that all the newly installed and upgraded libraries are correctly loaded into the environment.

#Import Libraries

In [ ]:
# ==============================================================================
# 2. Consolidated Library Imports
# ==============================================================================

# Standard Libraries
import os
import glob
import warnings

# Google Colab Environment Secrets
from google.colab import userdata, files
from typing import List, Dict

# Data & Array Processing
import pandas as pd
import numpy as np

# Core OpenAI client
from openai import OpenAI

# LangChain Ecosystem Components
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_experimental.text_splitter import SemanticChunker
from langchain_pinecone import PineconeVectorStore


from langchain_core.prompts import ChatPromptTemplate
from typing import TypedDict, List, Optional

from langchain_core.runnables import RunnablePassthrough

# LangGraph (State management)
from langgraph.graph import StateGraph, START, END
import langgraph

# Pinecone (Vector database client)
from pinecone import Pinecone, ServerlessSpec

# CrewAI (Multi-agent framework)
from crewai import Agent, Task, Crew, Process

# PDF Parsing Engine
import fitz  # PyMuPDF

# Suppress annoying deprecation/cleanliness warnings
warnings.filterwarnings("ignore")

print("✅ All core libraries and sub-modules imported successfully!")

In [ ]:
# !pip uninstall -y \
# langchain \
# langchain-core \
# langchain-community \
# langchain-classic \
# langchain-text-splitters

In [ ]:
# !pip install -U \
# langchain \
# langchain-core \
# langchain-community \
# langchain-classic \
# langchain-text-splitters

#Load API Keys

In [ ]:
# ============================================
# Load API Keys
# ============================================

import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

# Optional
try:
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
except:
    pass

print("✅ API keys loaded successfully!")

#Chunking

In [ ]:
#Perfoms the importing of the packages in requried for chunking. These pacakges were already imported but we do it her again for modulatity
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import PyPDFLoader

import os
import glob

In [ ]:
## Declares the OpenAIEmbeddings with text-embedding-3-small
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [ ]:
# Initializes a Semantic Chunker to split text based on 95th percentile shifts in embedding meaning
# Split the text when the difference between the sentences is having more than 95% difference

semantic_chunker = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95
)

In [ ]:
###This code loads all the content from all the PDF files tonto all_documents vairable. "Import glob" is essential for this code to work.
### We have done this already in the beginning of this notebook

pdf_folder = "documents"

all_documents = []

for pdf in glob.glob(f"{pdf_folder}/*.pdf"):
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    all_documents.extend(docs)

print(f"Loaded {len(all_documents)} pages.")

In [ ]:
###  Earlier we defined semantic_chunks ( with threshold type and amount).
## pass prepared "all_documents" into semantic_chunker. This will create semantic_chunks
## Means all the content loaded from all_documents will be split into meaningful chunks

semantic_chunks = semantic_chunker.split_documents(all_documents)


In [ ]:
# METADATA preparation : Loops through chunks to inject unique tracking IDs and clean document source names into metadata
# here if we dont mention .replace(".pdf", ""), when we retrive the data from vector store and look out for source,
# - it will display "product_catalog.pdf". This does not add any meaning. So, that is handled here.

for i, chunk in enumerate(semantic_chunks):
    chunk.metadata["chunk_id"] = i + 1
    chunk.metadata["document_type"] = (
        chunk.metadata["source"]
        .split("/")[-1]
        .replace(".pdf", "")
    )

In [ ]:
#### Just a display of what we have done in the previous steps and it shows the metadata
for chunk in semantic_chunks[:5]:
    print("=" * 80)
    print(f"Chunk ID      : {chunk.metadata['chunk_id']}")
    print(f"Source        : {chunk.metadata['source']}")
    print(f"Page          : {chunk.metadata['page']}")
    print(f"Document Type : {chunk.metadata['document_type']}")
    print("-" * 80)
    print(chunk.page_content[:500])

In [ ]:
# ============================================
# Enhance Chunk Metadata
# ============================================

### --> This section is just a manipulative/manual step for enhancing metadata.
### ---> Conceptual demonstration as how it can be done. In big production scale, we can have seperate agents designed to
# - extract the relevant keywords, important, product_list, document_ids etc.
# - It recursively identifies the category from each document and apply metadata

import os

# -----------------------------------------
# Product List
# -----------------------------------------

products = [
    "Ruby Solitaire Ring",
    "Emerald Halo Ring",
    "Diamond Eternity Ring",
    "Pearl Pendant Necklace",
    "Sapphire Tennis Bracelet",
    "Rose Gold Heart Pendant",
    "Diamond Stud Earrings",
    "Emerald Drop Earrings"
]

# -----------------------------------------
# Document IDs
# -----------------------------------------
DOC_IDS = {
    "Product_Catalog": "DOC001",
    "FAQ": "DOC002",
    "Return_Policy": "DOC003",
    "Shipping_Policy": "DOC004",
    "Warranty_Policy": "DOC005",
    "Jewellery_Care_Guide": "DOC006"
}

# -----------------------------------------
# Importance
# -----------------------------------------
IMPORTANCE = {
    "Product": "High",
    "Policy": "High",
    "FAQ": "Medium",
    "Care Guide": "Low",
    "General": "Low"
}

# -----------------------------------------
# Keywords
# -----------------------------------------
KEYWORDS = {

    "Product_Catalog": [
        "ruby","emerald","diamond","pearl","sapphire",
        "gold","white gold","yellow gold","rose gold",
        "silver","platinum","ring","bracelet",
        "necklace","earrings","solitaire","halo",
        "tennis bracelet","stud earrings","pendant"
    ],

    "Warranty_Policy": [
        "warranty","lifetime warranty","manufacturing defect",
        "repair","inspection","cleaning","loose gemstone",
        "clasp","scratches","chemical damage"
    ],

    "Return_Policy": [
        "return","exchange","refund","invoice",
        "original packaging","customized jewellery",
        "damaged product","business days"
    ],

    "Shipping_Policy": [
        "shipping","delivery","tracking",
        "metro cities","remote areas",
        "courier","free shipping",
        "shipping charges","dispatch"
    ],

    "FAQ": [
        "payment","credit card","debit card",
        "upi","emi","cash on delivery",
        "international shipping",
        "gift wrapping",
        "customer support"
    ],

    "Jewellery_Care_Guide": [
        "cleaning","storage","microfiber cloth",
        "soap","perfume","chlorine",
        "bleach","ultrasonic cleaner",
        "professional cleaning",
        "ring resizing"
    ]
}

# ============================================
# Process Each Chunk
# ============================================

for i, chunk in enumerate(semantic_chunks):

    # -----------------------------------------
    # Basic Information
    # -----------------------------------------
    source_file = os.path.basename(chunk.metadata.get("source", ""))
    document_name = source_file.replace(".pdf", "")
    document_id = DOC_IDS.get(document_name)

    # -----------------------------------------
    # Category
    # -----------------------------------------
    if "Product" in document_name:
        category = "Product"

    elif "FAQ" in document_name:
        category = "FAQ"

    elif "Return" in document_name:
        category = "Policy"

    elif "Shipping" in document_name:
        category = "Policy"

    elif "Warranty" in document_name:
        category = "Policy"

    elif "Care" in document_name:
        category = "Care Guide"

    else:
        category = "General"

    importance = IMPORTANCE.get(category)

    # -----------------------------------------
    # Product Name
    # -----------------------------------------
    product_name = None

    if category == "Product":
        for product in products:
            if product.lower() in chunk.page_content.lower():
                product_name = product
                break

    # -----------------------------------------
    # Section Detection
    # -----------------------------------------
    text = chunk.page_content.lower()

    if category == "Product":

        if "ruby solitaire ring" in text:
            section = "Ruby Solitaire Ring"

        elif "emerald halo ring" in text:
            section = "Emerald Halo Ring"

        elif "diamond eternity ring" in text:
            section = "Diamond Eternity Ring"

        elif "pearl pendant necklace" in text:
            section = "Pearl Pendant Necklace"

        elif "sapphire tennis bracelet" in text:
            section = "Sapphire Tennis Bracelet"

        elif "rose gold heart pendant" in text:
            section = "Rose Gold Heart Pendant"

        elif "diamond stud earrings" in text:
            section = "Diamond Stud Earrings"

        elif "emerald drop earrings" in text:
            section = "Emerald Drop Earrings"

        else:
            section = "Product Overview"

    elif document_name == "Warranty_Policy":

        if "lifetime warranty" in text:
            section = "Lifetime Warranty"

        elif "standard warranty" in text:
            section = "Standard Warranty"

        elif "warranty covers" in text:
            section = "Warranty Coverage"

        elif "does not cover" in text:
            section = "Warranty Exclusions"

        elif "cleaning" in text:
            section = "Cleaning Service"

        else:
            section = "Warranty"

    elif document_name == "Return_Policy":

        if "refund" in text:
            section = "Refund Policy"

        elif "exchange" in text:
            section = "Exchange Policy"

        else:
            section = "Return Policy"

    elif document_name == "Shipping_Policy":

        if "shipping charges" in text:
            section = "Shipping Charges"

        elif "delivery time" in text:
            section = "Delivery Time"

        elif "tracking" in text:
            section = "Order Tracking"

        else:
            section = "Shipping"

    elif document_name == "Jewellery_Care_Guide":

        if "professional cleaning" in text:
            section = "Professional Cleaning"

        elif "storage" in text:
            section = "Storage"

        elif "cleaning" in text:
            section = "Cleaning"

        elif "avoid" in text:
            section = "Precautions"

        else:
            section = "Jewellery Care"

    elif category == "FAQ":
        section = "Frequently Asked Questions"

    else:
        section = "General"

    # -----------------------------------------
    # Keywords
    # -----------------------------------------
    keywords = []

    for keyword in KEYWORDS.get(document_name, []):
        if keyword.lower() in text:
            keywords.append(keyword)

    if product_name:
        keywords.append(product_name)

    keywords = list(set(keywords))

    # -----------------------------------------
    # Remove Unnecessary Metadata
    # -----------------------------------------
    for field in [
        "producer",
        "creator",
        "creationdate",
        "author",
        "moddate"
    ]:
        chunk.metadata.pop(field, None)

    # -----------------------------------------
    # Update Metadata
    # -----------------------------------------
    chunk.metadata.update({

        "chunk_id": i + 1,
        "document_id": document_id,
        "document_type": document_name,
        "document_name": document_name,
        "source_file": source_file,
        "category": category,
        "importance": importance,
        "section": section,
        "product_name": product_name,
        "keywords": keywords,
        "page_number": chunk.metadata.get("page", 0)

    })

print(f"✅ Metadata enhanced for {len(semantic_chunks)} chunks.")

In [ ]:
## Not relevant. This is just for validation
semantic_chunks[2].metadata

In [ ]:
## Not relevant. This is just for validation
for chunk in semantic_chunks[:25]:
    print("=" * 80)
    print(chunk.metadata)

In [ ]:
## Not relevant. This is just for validation
print(f"Total Chunks: {len(semantic_chunks)}")

In [ ]:
## Not relevant. This is just for validation

sample_text = "Returns are accepted within 30 days."

embedding = embeddings.embed_query(sample_text)

print(f"Embedding Dimension: {len(embedding)}")
print(embedding[:10])   # Print first 10 values

In [ ]:
## Not relevant. This is just for validation
print(semantic_chunks[0].page_content)

#Embedding

In [ ]:
###Just displays the top chunk, total number of chunks and metadata.
print(f"Total chunks: {len(semantic_chunks)}")

# Display the first chunk
print("\nFirst Chunk:")
print(semantic_chunks[0].page_content[:300])

print("\nMetadata:")
print(semantic_chunks[0].metadata)

In [ ]:
#### Extracts only the raw text content from the semantic chunks into a clean list of strings NS lists them
texts = [doc.page_content for doc in semantic_chunks]
print("texts")

In [ ]:
###It strips the spaces, new lines and loads the semantic_chunks. Displays semantic_chunk adn filter_chunk counts
filtered_chunks = [
    doc for doc in semantic_chunks
    if doc.page_content.strip()
]

print(f"Original chunks : {len(semantic_chunks)}")
print(f"Filtered chunks : {len(filtered_chunks)}")

In [ ]:
#### Extracts only the raw text content from the semantic chunks
texts = [doc.page_content for doc in filtered_chunks]

In [ ]:
### Convertint the raw text extracted from filtered_chunks into vectors which will be used in pinecone vector update
vectors = embeddings.embed_documents(texts)

In [ ]:
###Only for validation
print(f"Number of chunks     : {len(texts)}")
print(f"Number of embeddings : {len(vectors)}")

In [ ]:
###Only for validation
print(f"Embedding dimension: {len(vectors[0])}")

In [ ]:
###Only for validation
print(vectors[0][:10])

In [ ]:
###Only for validation
for i in range(3):
    print("=" * 60)
    print(f"Chunk {i}")
    print(filtered_chunks[i].metadata)
    print(filtered_chunks[i].page_content[:150])

In [ ]:
###Only for validation
assert len(filtered_chunks) == len(vectors), \
    "Mismatch between chunks and embeddings."

assert len(vectors[0]) == 1536, \
    "Unexpected embedding dimension."

print("Embedding generation completed successfully.")

In [ ]:
###Only for validation
print(type(texts))
print(type(vectors))
print(type(filtered_chunks))

#Connect to Pinecone

In [ ]:
##Gets pinecone API
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

In [ ]:
###Assign the retrieved api_key to the variable pc
pc = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)

In [ ]:
###The same can be configured manually in pinecone console
INDEX_NAME = "customer-support-agent"
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIMENSION = 1536
METRIC = "cosine"
CLOUD_PROVIDER = "aws"
REGION = "us-east-1"

In [ ]:
###This is just for evaluation
existing_indexes = pc.list_indexes().names()
print(existing_indexes)

In [ ]:
####Creates the index with the previously established configuration

if INDEX_NAME not in existing_indexes:

    print(f"Creating index: {INDEX_NAME}")

    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(
            cloud=CLOUD_PROVIDER,
            region=REGION
        )
    )

else:

    print(f"Index '{INDEX_NAME}' already exists.")

In [ ]:
#### A safety mehachnism to wait sometime before checking the availability of the index
while not pc.describe_index(INDEX_NAME).status["ready"]:
    print("Waiting for index to become ready...")
    time.sleep(5)

print("Index is ready.")

In [ ]:
#### A validation step
index = pc.Index(INDEX_NAME)
print("Connected successfully.")

In [ ]:
#### Validation step
index_info = pc.describe_index(INDEX_NAME)
print(index_info)

In [ ]:
#### A validation step
stats = index.describe_index_stats()
print(stats)

In [ ]:
#### This is for reproducability. Not actually used in the note book later
PINECONE_CONFIG = {
    "index_name": INDEX_NAME,
    "embedding_model": EMBEDDING_MODEL,
    "dimension": EMBEDDING_DIMENSION,
    "metric": METRIC,
    "cloud": CLOUD_PROVIDER,
    "region": REGION
}
print(PINECONE_CONFIG)

In [ ]:
#### A validation step
print(f"Texts      : {len(texts)}")
print(f"Vectors    : {len(vectors)}")
print(f"Documents  : {len(filtered_chunks)}")

assert len(texts) == len(vectors) == len(filtered_chunks)

In [ ]:
#### A validation step
vector_ids = [
    f"chunk_{i:05d}"
    for i in range(len(filtered_chunks))
]

print(vector_ids[:5])

In [ ]:
####Handles if there is any None values in metadata and prepares a clean metadata
clean_metadata = []
for doc in filtered_chunks:
    metadata = {}
    for key, value in doc.metadata.items():
        if value is None:
            continue
        metadata[key] = str(value)
    clean_metadata.append(metadata)

In [ ]:
#### A validation step
print(clean_metadata[0])

In [ ]:
#### A validation step
for metadata, text in zip(clean_metadata, texts):

    metadata["text"] = text

In [ ]:
#### A validation step
print(clean_metadata[0].keys())

In [ ]:
###This is where pinecone records are prepared
pinecone_records = []

for vector_id, embedding, metadata in zip(
    vector_ids,
    vectors,
    clean_metadata
):

    pinecone_records.append(
        (
            vector_id,
            embedding,
            metadata
        )
    )

In [ ]:
#### A validation step
print(pinecone_records[0][0])
print(len(pinecone_records[0][1]))
print(pinecone_records[0][2])

In [ ]:
### Config parameter is set
BATCH_SIZE = 100

In [ ]:
#### This is where the prepared pinecond_records are upserted into pinecone
from math import ceil

total_batches = ceil(len(pinecone_records) / BATCH_SIZE)

for batch_number in range(total_batches):

    start = batch_number * BATCH_SIZE
    end = start + BATCH_SIZE

    batch = pinecone_records[start:end]

    index.upsert(vectors=batch)

    print(
        f"Uploaded batch "
        f"{batch_number + 1}/{total_batches}"
    )

In [ ]:
#### A validation step
stats = index.describe_index_stats()

print(stats)

In [ ]:
#### A validation step
uploaded_vectors = stats["total_vector_count"]

expected_vectors = len(filtered_chunks)

print(f"Uploaded : {uploaded_vectors}")
print(f"Expected : {expected_vectors}")

assert uploaded_vectors == expected_vectors

In [ ]:
#### A validation step
query_vector = vectors[0]

results = index.query(
    vector=query_vector,
    top_k=3,
    include_metadata=True
)

print(results)

In [ ]:
####### A configuration reproducable module is prepared. Not used elsewhere though
RAG_CONFIG = {

    "index_name": INDEX_NAME,

    "embedding_model": EMBEDDING_MODEL,

    "dimension": EMBEDDING_DIMENSION,

    "metric": METRIC,

    "batch_size": BATCH_SIZE
}

print(RAG_CONFIG)

#Build and Test the Semantic Retriever
Objective

In [ ]:
### Embeddings are set
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [ ]:
#### A validation step
pc = Pinecone(
    api_key=os.environ["PINECONE_API_KEY"]
)

INDEX_NAME = "customer-support-agent"

index = pc.Index(INDEX_NAME)

In [ ]:
#### A declaring the vector store
vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings
)

In [ ]:
#### A validation step
stats = index.describe_index_stats()

print(stats)

In [ ]:
#### Retriever is declared
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k":3
    }
)

In [ ]:
### Passing the query through retreiver connects to pinecone to produce the results
#### A validation step
query = "What is the warranty period for diamond jewellery?"

results = retriever.invoke(query)

In [ ]:
#### A validation step
print(f"Retrieved {len(results)} documents\n")

for i, doc in enumerate(results, start=1):

    print("=" * 80)
    print(f"Result {i}")
    print("=" * 80)

    print("\nMetadata:")
    print(doc.metadata)

    print("\nContent:")
    print(doc.page_content[:500])

    print("\n")

In [ ]:
#### A validation step
test_queries = [

    "How long is the warranty?",

    "Can I return a customised ring?",

    "How many days does shipping take?",

    "How should I clean my jewellery?",

    "What if my ring arrives damaged?"

]

In [ ]:
#### A validation step
for query in test_queries:

    print("=" * 80)
    print(query)
    print("=" * 80)

    docs = retriever.invoke(query)

    print(docs[0].page_content[:300])

    print("\n")

In [ ]:
#### A reproducable retriever configuration setup. Not used elsewhere
RETRIEVER_CONFIG = {

    "embedding_model": "text-embedding-3-small",

    "index_name": INDEX_NAME,

    "search_type": "similarity",

    "top_k": 5

}

print(RETRIEVER_CONFIG)

#Product Catalog Processing

In [ ]:
#Loading the Product_Catalog.pdf for later processing
from langchain_community.document_loaders import PyPDFLoader  #to parese and hald pdf

loader = PyPDFLoader("documents/Product_Catalog.pdf")

pages = loader.load()

catalog_text = "\n".join(page.page_content for page in pages)

In [ ]:
#Importing regular expressions to handle the product_catlog doucment into splits & removes accidental spaces and new lines

import re

products = re.split(
    r"\n(?=\d+\.\s)",
    catalog_text
)

products = [p.strip() for p in products if p.strip()]

In [ ]:
###Prepares the document and makes product_documents which will be later used for embedding
from langchain_core.documents import Document

product_documents = []

for i, product in enumerate(products):

    lines = product.split("\n")

    product_name = lines[0].replace(".", "", 1).strip()

    doc = Document(

        page_content=product,

        metadata={

            "document_name": "Product_Catalog",

            "category": "Product",

            "product_name": product_name,

            "chunk_id": f"product_{i+1}",

            "importance": "High"

        }

    )

    product_documents.append(doc)

In [ ]:
### # This embeds the documents and stores them in the vector database
vector_store.add_documents(product_documents)

#Build RAG Pipeline

In [ ]:
#### LLM is setup
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=700
)

In [ ]:
#### #### This used in retriever to fetch the documents when the RAG chain is called
def format_documents(documents):

    return "\n\n".join(
        doc.page_content
        for doc in documents
    )

In [ ]:
#### System Prompt for RAG LLM
SYSTEM_PROMPT = """
You are Ornativa's AI Customer Support Assistant.

Your responsibilities:

1. Answer ONLY using the provided context.
2. Never invent or assume information.
3. If the answer is not present in the context, clearly say:
   "I couldn't find that information in the available documents."
4. Be concise, accurate and professional.
5. When possible, mention the relevant product or policy.
6. If a product is unavailable in the provided context, state that you cannot confirm its availability.
7. Never mention internal implementation details.
8. If multiple retrieved passages contain relevant information, combine them into a single coherent answer.

Context:
{context}

Customer Question:
{question}
"""

In [ ]:
######Prompt is prepared with System_Prompt
prompt = ChatPromptTemplate.from_template(

    SYSTEM_PROMPT
)

In [ ]:
#### RAG Chain is defined
rag_chain = (

    {
        "context": retriever | format_documents,

        "question": RunnablePassthrough()
    }

    | prompt

    | llm

    | StrOutputParser()

)

In [ ]:
##### A validation step
test_questions = [
    "What is the warranty period?",
    "Can I return customised jewellery?",
    "How long does shipping take?",
    "How do I clean a diamond ring?",
    "What should I do if I receive a damaged product?",
    "Do you offer lifetime warranty?"
]

for test_question in test_questions:

    print("=" * 80)
    print(test_question)
    print("=" * 80)

    response = rag_chain.invoke(test_question)

    print(response)
    print()

In [ ]:
#### A question call functionality connects, retriever for questions, uses it for context and anwers with rag_chain
def ask_question(question):

    docs = retriever.invoke(question)

    context = format_documents(docs)

    answer = rag_chain.invoke(question)

    return {

        "question": question,

        "answer": answer,

        "documents": docs,

        "context": context

    }

In [ ]:
##### A validation step
result = ask_question(
    "List all products with price?"
)

print(result["answer"])

print("\nSources:\n")

for doc in result["documents"]:

    print(doc.metadata)

In [ ]:
####  Areproducable module. Not used elsewhere
RAG_CONFIG = {

    "llm": "gpt-4o-mini",

    "embedding_model": "text-embedding-3-small",

    "retriever": "similarity",

    "top_k": 5,

    "temperature": 0,

    "max_tokens": 700
}

print(RAG_CONFIG)

#Customer Support Agent

 Below are just experimentation with Langraph

In [ ]:
#### An adhoc insert
import importlib.metadata

version = importlib.metadata.version("langgraph")
print(version)

In [ ]:
#### Initial Customer Support Class declaration
class CustomerSupportState(TypedDict):
    """
    Shared state passed between LangGraph nodes.
    """

    question: str

    documents: List

    context: str

    answer: str

    source_documents: List

    status: str


    error: Optional[str]

In [ ]:
#### Definiting Initialize Agent
def initialize_agent(
    state: CustomerSupportState,
) -> CustomerSupportState:
    """
    Initialize the customer support workflow.
    """

    print("\n========== Initialize Agent ==========")
    print("History received:", len(state.get("conversation_history", [])))

    question = state["question"].strip()

    return {
        **state,
        "question": question,
        "documents": [],
        "context": "",
        "answer": "",
        "source_documents": [],
        "conversation_history": state.get("conversation_history", []),
        "status": "initialized",
        "error": None,
    }

In [ ]:
#### Initializing retrieval agent
def retrieve_documents(state: CustomerSupportState):

    print("\n========== Retrieval Agent ==========")

    #query = state["rewritten_question"]
    query = state["question"]

    print(f"Searching for: {query}")

    documents = retriever.invoke(query)

    context = format_documents(documents)

    print(f"Retrieved {len(documents)} document(s).")

    return {
        **state,
        "documents": documents,
        "context": context,
        "status": "retrieved"
    }

In [ ]:
#### Langgraph buildign with initialize and retrieval again
builder = StateGraph(CustomerSupportState)

builder.add_node("initialize_agent", initialize_agent)
builder.add_node("retrieve_documents", retrieve_documents)

builder.add_edge(START, "initialize_agent")
builder.add_edge("initialize_agent", "retrieve_documents")
builder.add_edge("retrieve_documents", END)

customer_support_graph = builder.compile()

In [ ]:
#### Declaring initial_State
initial_state = {
    "question": "Does the Diamond Eternity Ring include a warranty?",
    "documents": [],
    "context": "",
    "answer": "",
    "source_documents": [],
    "status": "",
    "error": None
}

In [ ]:
#### invoking the graph with initial state
result = customer_support_graph.invoke(initial_state)

In [ ]:
##### Prompt for Response Agent
from langchain_core.prompts import ChatPromptTemplate

response_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI Customer Support Assistant for a jewellery company.

Your responsibility is ONLY to answer the customer's question using the retrieved context.

Rules:

1. Use ONLY the retrieved context to generate your answer.
2. Never hallucinate or make up information.
3. If the answer is unavailable, politely state that the information is not available in the provided documents.
4. If the requested product is unavailable or out of stock, clearly inform the customer that it is unavailable.
5. Do NOT recommend alternative products.
6. Do NOT suggest similar products.
7. Do NOT provide cross-sell or up-sell suggestions.
8. Recommendations are handled by a separate Recommendation Agent.
9. Keep the response concise, professional, and customer-friendly.
10. Mention source information naturally only when it helps answer the customer's question.
11.If the customer question contains a product name, answer ONLY using that product.
12.Ignore every other product in the retrieved context.
13.Do not compare products unless the user explicitly asks.
14.Never answer using information from another product.
"""
        ),
        (
            "human",
            """
Customer Question:

{question}

Retrieved Context:

{context}

Answer:
"""
        )
    ]
)

In [ ]:
#### Response Chain creation
response_chain = (
    response_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
### Response Agent
def generate_response(state: CustomerSupportState):
    """
    Generate the final response from the retrieved context
    and update the conversation history.
    """

    print("\n========== Response Agent ==========")
    print("History:", len(state.get("conversation_history", [])))

    try:

        # No relevant context retrieved
        if not state["context"]:

            answer = (
                "I'm sorry, but I couldn't find any relevant information in the knowledge base."
            )

            # Retrieve existing history
            conversation_history = state.get("conversation_history", []).copy()

            # Save user message
            conversation_history.append(
                {
                    "role": "user",
                    "content": state["question"]
                }
            )

            # Save assistant response
            conversation_history.append(
                {
                    "role": "assistant",
                    "content": answer
                }
            )

            return {
                **state,
                "answer": answer,
                "conversation_history": conversation_history,
                "status": "no_context"
            }

        # Generate answer using the LLM
        answer = response_chain.invoke(
            {
                "question": state["question"],
                "context": state["context"]
            }
        )

        print(f"Generated Answer:\n{answer}")

        # Retrieve existing history
        conversation_history = state.get("conversation_history", []).copy()

        # Save user message
        conversation_history.append(
            {
                "role": "user",
                "content": state["question"]
            }
        )

        # Save assistant response
        conversation_history.append(
            {
                "role": "assistant",
                "content": answer
            }
        )

        return {
            **state,
            "answer": answer,
            "conversation_history": conversation_history,
            "status": "answered"
        }

    except Exception as e:

        return {
            **state,
            "status": "failed",
            "error": str(e)
        }

In [ ]:
#### Update Lang Graph with response agent

builder = StateGraph(CustomerSupportState)

builder.add_node("initialize_agent", initialize_agent)
builder.add_node("retrieve_documents", retrieve_documents)
builder.add_node("generate_response", generate_response)

builder.add_edge(START, "initialize_agent")
builder.add_edge( "initialize_agent", "retrieve_documents")
builder.add_edge( "retrieve_documents", "generate_response")
builder.add_edge("generate_response", END)

customer_support_graph = builder.compile()

In [ ]:
response_chain = (
    response_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
#### Declaring initial_State
initial_state = {
    "question": "Does the Diamond Eternity Ring include a warranty?",
    "documents": [],
    "context": "",
    "answer": "",
    "source_documents": [],
    "status": "",
    "error": None
}

In [ ]:
###invoking the graph with initial_state
result = customer_support_graph.invoke(initial_state)

In [ ]:
### Validation party
print(result["answer"])

In [ ]:
####Intent classification part
# Declaring the CustomerSupportState

from typing import Literal

class CustomerSupportState(TypedDict):
    question: str

    intent: str

    documents: list

    context: str

    answer: str

    source_documents: list

    status: str

    error: str | None

In [ ]:
###Create intent prompt

from langchain_core.prompts import ChatPromptTemplate

intent_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an intent classification agent.

Classify the customer's question into exactly ONE category.

Valid categories:

PRODUCT
WARRANTY
RETURNS
SHIPPING
CARE
RECOMMENDATION
GENERAL
UNKNOWN

Return ONLY the category name.
"""
        ),

        (
            "human",
            "{question}"
        )
    ]
)

In [ ]:
response_parser = StrOutputParser()

In [ ]:
intent_chain = (
    intent_prompt
    | llm
    | response_parser
)

In [ ]:
### Intent Classification Agent
def classify_intent(state: CustomerSupportState):
    """
    Classify the customer's intent.
    """

    print("\n========== Intent Agent ==========")
    print("History:", len(state.get("conversation_history", [])))

    try:

        intent = intent_chain.invoke(
            {
                "question": state["question"]
            }
        ).strip().upper()

        valid_intents = {
            "PRODUCT",
            "WARRANTY",
            "RETURNS",
            "SHIPPING",
            "CARE",
            "RECOMMENDATION",
            "GENERAL",
            "UNKNOWN"
        }

        if intent not in valid_intents:
            intent = "UNKNOWN"

        print(f"Intent: {intent}")

        return {
            **state,
            "intent": intent,
            "status": "intent_classified"
        }

    except Exception as e:

        return {
            **state,
            "intent": "UNKNOWN",
            "status": "intent_failed",
            "error": str(e)
        }

In [ ]:
#### Declaring initial_state
initial_state = {
    "question": "Does the Diamond Eternity Ring include a warranty?",

    "intent": "",

    "documents": [],

    "context": "",

    "answer": "",

    "source_documents": [],

    "status": "",

    "error": None
}

In [ ]:
#### Updating the graph
builder = StateGraph(CustomerSupportState)

builder.add_node("initialize_agent", initialize_agent)

builder.add_node("classify_intent", classify_intent)

builder.add_node("retrieve_documents", retrieve_documents)

builder.add_node("generate_response", generate_response)


builder.add_edge(
    START,
    "initialize_agent"
)

builder.add_edge(
    "initialize_agent",
    "classify_intent"
)

builder.add_edge(
    "classify_intent",
    "retrieve_documents"
)

builder.add_edge(
    "retrieve_documents",
    "generate_response"
)

builder.add_edge(
    "generate_response",
    END
)

customer_support_graph = builder.compile()

In [ ]:
#### invoking with initial_state and validation
result = customer_support_graph.invoke(initial_state)

print(result["intent"])

print(result["answer"])

In [ ]:
#### Router Constants
ROUTE_PRODUCT = "product"

ROUTE_WARRANTY = "warranty"

ROUTE_RETURNS = "returns"

ROUTE_SHIPPING = "shipping"

ROUTE_CARE = "care"

ROUTE_RECOMMENDATION = "recommendation"

ROUTE_GENERAL = "general"

ROUTE_UNKNOWN = "unknown"

In [ ]:
###Create Router Function
def route_intent(state: CustomerSupportState):
    """
    Determine the next workflow based on intent.
    """

    intent = state.get("intent", "UNKNOWN")

    routing_table = {

        "PRODUCT": ROUTE_PRODUCT,

        "WARRANTY": ROUTE_WARRANTY,

        "RETURNS": ROUTE_RETURNS,

        "SHIPPING": ROUTE_SHIPPING,

        "CARE": ROUTE_CARE,

        "RECOMMENDATION": ROUTE_RECOMMENDATION,

        "GENERAL": ROUTE_GENERAL

    }

    route = routing_table.get(intent, ROUTE_UNKNOWN)

    print(f"Routing → {route}")

    return route

In [ ]:
###Create Placeholder Nodes
def product_workflow(state):
    return state


def warranty_workflow(state):
    return state


def shipping_workflow(state):
    return state


def returns_workflow(state):
    return state


def care_workflow(state):
    return state


def recommendation_workflow(state):
    return state


def general_workflow(state):
    return state


def unknown_workflow(state):
    return state

In [ ]:
###Register Nodes
builder.add_node("product", product_workflow)

builder.add_node("warranty", warranty_workflow)

builder.add_node("shipping", shipping_workflow)

builder.add_node("returns", returns_workflow)

builder.add_node("care", care_workflow)

builder.add_node("recommendation", recommendation_workflow)

builder.add_node("general", general_workflow)

builder.add_node("unknown", unknown_workflow)

In [ ]:
### Add conditional Edges
builder.add_conditional_edges(
    "classify_intent",

    route_intent,

    {

        ROUTE_PRODUCT: "product",

        ROUTE_WARRANTY: "warranty",

        ROUTE_RETURNS: "returns",

        ROUTE_SHIPPING: "shipping",

        ROUTE_CARE: "care",

        ROUTE_RECOMMENDATION: "recommendation",

        ROUTE_GENERAL: "general",

        ROUTE_UNKNOWN: "unknown"

    }


    )

In [ ]:
###Connect the placeholder nodes
builder.add_edge("product", "retrieve_documents")

builder.add_edge("warranty", "retrieve_documents")

builder.add_edge("shipping", "retrieve_documents")

builder.add_edge("returns", "retrieve_documents")

builder.add_edge("care", "retrieve_documents")

builder.add_edge("recommendation", "retrieve_documents")

builder.add_edge("general", "retrieve_documents")



builder.add_edge("unknown", "retrieve_documents")

In [ ]:
# Register all nodes and build graph
builder = StateGraph(CustomerSupportState)

builder.add_node("initialize_agent", initialize_agent)
builder.add_node("classify_intent", classify_intent)
builder.add_node("product", product_workflow)
builder.add_node("warranty", warranty_workflow)
builder.add_node("shipping", shipping_workflow)
builder.add_node("returns", returns_workflow)
builder.add_node("care", care_workflow)
builder.add_node("recommendation", recommendation_workflow)
builder.add_node("general", general_workflow)
builder.add_node("unknown", unknown_workflow)
builder.add_node("retrieve_documents", retrieve_documents)
builder.add_node("generate_response", generate_response)

# Add edges
builder.add_edge(START, "initialize_agent")
builder.add_edge("initialize_agent", "classify_intent")

builder.add_conditional_edges(
    "classify_intent",
    route_intent,
    {
        ROUTE_PRODUCT: "product",
        ROUTE_WARRANTY: "warranty",
        ROUTE_RETURNS: "returns",
        ROUTE_SHIPPING: "shipping",
        ROUTE_CARE: "care",
        ROUTE_RECOMMENDATION: "recommendation",
        ROUTE_GENERAL: "general",
        ROUTE_UNKNOWN: "unknown",
    },
)

builder.add_edge("product", "retrieve_documents")
builder.add_edge("warranty", "retrieve_documents")
builder.add_edge("shipping", "retrieve_documents")
builder.add_edge("returns", "retrieve_documents")
builder.add_edge("care", "retrieve_documents")
builder.add_edge("recommendation", "retrieve_documents")
builder.add_edge("general", "retrieve_documents")
builder.add_edge("unknown", "retrieve_documents")

builder.add_edge("retrieve_documents", "generate_response")
builder.add_edge("generate_response", END)

customer_support_graph = builder.compile()

In [ ]:
###Validation
initial_state["question"] = "Does this ring include warranty?"

result = customer_support_graph.invoke(initial_state)

print(result["intent"])

In [ ]:
###Validation
initial_state["question"] = "How long is shipping?"

result = customer_support_graph.invoke(initial_state)

print(result["intent"])

In [ ]:
###Validation
initial_state["question"] = "What is the return policy?"

result = customer_support_graph.invoke(initial_state)

print(result["intent"])

In [ ]:
######QUERY REWRITE

In [ ]:
### Updated Class for CustomerSupportState
class CustomerSupportState(TypedDict):
    question: str

    rewritten_question: str

    intent: str

    documents: list

    context: str

    answer: str

    source_documents: list

    status: str

    validation_status: str

    validated_answer: str

    error: str | None

In [ ]:
### REWRITE PROMPT

from langchain_core.prompts import ChatPromptTemplate

rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert query rewriting assistant for a jewellery customer support RAG system.

Your task is to rewrite the current user question into a complete, standalone question that maximizes semantic retrieval accuracy.

Conversation History:
{history}

Current User Question:
{question}

Instructions:

1. Read the conversation history carefully before rewriting.

2. If the current question contains pronouns or ambiguous references such as:
   - it
   - its
   - this
   - that
   - they
   - them
   - one
   - first one
   - second one
   - cheaper one
   - expensive one
   - available one
   - same one

   determine exactly which product the user is referring to from the conversation history.

3. Replace every ambiguous reference with the COMPLETE product name.

4. NEVER rewrite using generic words such as:
   - item
   - product
   - jewellery
   - accessory
   - ring
   - bracelet
   unless those words are part of the official product name.

5. If the previous conversation clearly discussed exactly one product, always use that product name.

6. If multiple products appear in the conversation history, resolve the reference to the product that was the PRIMARY subject of the assistant's most recent answer, not products mentioned only as recommendations.

7. Do NOT invent any product names.

8. Preserve the user's original intent.

9. If the current question is already complete and unambiguous, return it unchanged.

Examples

Conversation:
User: Tell me about the Sapphire Tennis Bracelet.
Assistant: The Sapphire Tennis Bracelet is made of 18K White Gold and is currently in stock.

Question:
Is it in stock?

Output:
Is the Sapphire Tennis Bracelet in stock?

----------------------------------------

Conversation:
User: Show me the Emerald Halo Ring.
Assistant: The Emerald Halo Ring costs ₹42,999.

Question:
What is its warranty?

Output:
What is the warranty of the Emerald Halo Ring?

----------------------------------------

Conversation:
User: Which rings are available?
Assistant: Ruby Solitaire Ring is out of stock.
Emerald Halo Ring is in stock.

Question:
Which one is cheaper?

Output:
Which product is cheaper: Ruby Solitaire Ring or Emerald Halo Ring?

Return ONLY the rewritten question.
"""
        ),
        (
            "human",
            "{question}"
        )
    ]
)

In [ ]:
rewrite_chain = (
    rewrite_prompt
    | llm
    | response_parser
)

In [ ]:
###QUERY REWRITER AGENT
def rewrite_query(state: CustomerSupportState):
    """
    Rewrite the customer question before retrieval.
    """


    print("\n========== Query Rewriter ==========")
    print("History:", len(state.get("conversation_history", [])))

    ###Newly added

    history = state.get("conversation_history", [])

    history_text = "\n".join(
    f"{msg['role'].capitalize()}: {msg['content']}"
    for msg in history
    )

    print("\n----- DEBUG -----")
    print("Question:", state["question"])
    print("History length:", len(history))
    print("History:")
    print(history_text)
    print("-----------------\n")

    ########
    try:

        rewritten = rewrite_chain.invoke(
            {
                "question": state["question"],
                "history": history_text
            }
        ).strip()

        print(f"Original : {state['question']}")
        print(f"Rewritten: {rewritten}")

        return {
            **state,
            "rewritten_question": rewritten,
            "status": "query_rewritten"
        }

    except Exception as e:

        return {
            **state,
            "rewritten_question": state["question"],
            "status": "rewrite_failed",
            "error": str(e)
        }

In [ ]:
builder = StateGraph(CustomerSupportState)

# Register all nodes
builder.add_node("initialize_agent", initialize_agent)
builder.add_node("classify_intent", classify_intent)
builder.add_node("product", product_workflow)
builder.add_node("warranty", warranty_workflow)
builder.add_node("shipping", shipping_workflow)
builder.add_node("returns", returns_workflow)
builder.add_node("care", care_workflow)
builder.add_node("recommendation", recommendation_workflow)
builder.add_node("general", general_workflow)
builder.add_node("unknown", unknown_workflow)
builder.add_node("retrieve_documents", retrieve_documents)
builder.add_node("generate_response", generate_response)
builder.add_node("rewrite_query", rewrite_query)

# Add edges
builder.add_edge(START, "initialize_agent")
builder.add_edge("initialize_agent", "classify_intent")
builder.add_edge("classify_intent", "rewrite_query")

builder.add_conditional_edges(
    "rewrite_query",
    route_intent,
    {
        ROUTE_PRODUCT: "product",
        ROUTE_WARRANTY: "warranty",
        ROUTE_RETURNS: "returns",
        ROUTE_SHIPPING: "shipping",
        ROUTE_CARE: "care",
        ROUTE_RECOMMENDATION: "recommendation",
        ROUTE_GENERAL: "general",
        ROUTE_UNKNOWN: "unknown",
    },
)

builder.add_edge("product", "retrieve_documents")
builder.add_edge("warranty", "retrieve_documents")
builder.add_edge("shipping", "retrieve_documents")
builder.add_edge("returns", "retrieve_documents")
builder.add_edge("care", "retrieve_documents")
builder.add_edge("recommendation", "retrieve_documents")
builder.add_edge("general", "retrieve_documents")
builder.add_edge("unknown", "retrieve_documents")

builder.add_edge("retrieve_documents", "generate_response")
builder.add_edge("generate_response", END)

customer_support_graph = builder.compile()

In [ ]:
def is_catalogue_query(query: str) -> bool:

    query = query.lower()

    catalogue_keywords = [

        "list all products",
        "show all products",
        "show catalogue",
        "show catalog",
        "what products",
        "available products",
        "display products",
        "list products"

    ]

    return any(keyword in query for keyword in catalogue_keywords)

In [ ]:
# retrieval
# Retrieval Agent
def retrieve_documents(state: CustomerSupportState):

    print("\n========== Retrieval Agent ==========")
    print("History:", len(state.get("conversation_history", [])))

    query = state["rewritten_question"]
    intent = state["intent"]

    print(f"Intent    : {intent}")
    print(f"Searching : {query}")

    # --------------------------------------------------
    # Number of chunks to retrieve
    # --------------------------------------------------
    if intent == "PRODUCT":
        k = 2
    elif intent == "RECOMMENDATION":
        k = 6
    elif intent in ["GENERAL", "UNKNOWN"]:
        k = 6
    else:
        k = 5

    # --------------------------------------------------
    # PRODUCT
    # --------------------------------------------------
    if intent == "PRODUCT":

        # Handle catalogue requests separately
        if is_catalogue_query(query):

            print("Catalogue query detected.")

            documents = vector_store.similarity_search(
                query="Product Catalogue",
                k=100,
                filter={
                    "document_name": "Product_Catalog"
                }
            )

        # Normal product search
        else:

            documents = vector_store.similarity_search(
                query=query,
                k=k,
                filter={
                    "document_name": "Product_Catalog"
                }
            )

    # --------------------------------------------------
    # WARRANTY
    # --------------------------------------------------
    elif intent == "WARRANTY":

        documents = vector_store.similarity_search(
            query=query,
            k=k,
            filter={
                "document_name": "Warranty_Policy"
            }
        )

    # --------------------------------------------------
    # SHIPPING
    # --------------------------------------------------
    elif intent == "SHIPPING":

        documents = vector_store.similarity_search(
            query=query,
            k=k,
            filter={
                "document_name": "Shipping_Policy"
            }
        )

    # --------------------------------------------------
    # GENERAL / UNKNOWN / RECOMMENDATION / OTHERS
    # --------------------------------------------------
    else:

        documents = vector_store.similarity_search(
            query=query,
            k=k
        )

    # --------------------------------------------------
    # Context Ranking
    # --------------------------------------------------

    # Rank retrieved documents by metadata importance
    documents = sorted(
        documents,
        key=lambda doc: doc.metadata.get("importance", 0),
        reverse=True
    )

    # Number of chunks to pass to the LLM
    if intent == "PRODUCT" and is_catalogue_query(query):
        TOP_K_CONTEXT = 20
    else:
        TOP_K_CONTEXT = 5

    documents = documents[:TOP_K_CONTEXT]

    print(f"\nRetrieved {len(documents)} document(s).")

    print("\n========== Retrieved Documents ==========")

    for i, doc in enumerate(documents, start=1):
        print(f"\nDocument {i}")
        print("Source :", doc.metadata.get("document_name"))
        print(doc.page_content[:250])
        print("-" * 60)

    context = format_documents(documents)

    return {
        **state,
        "documents": documents,
        "context": context,
        "status": "retrieved"
    }

In [ ]:
initial_state = {
    "question": "Does the Diamond Eternity Ring include a warranty?",

    "intent": "",

    "documents": [],

    "context": "",

    "answer": "",

    "source_documents": [],

    "status": "",

    "validated_answer": "",

    "validation_status": "",

    "error": None
}

In [ ]:
###Validation Prompt
from langchain_core.prompts import ChatPromptTemplate

validation_prompt = ChatPromptTemplate.from_template(
"""
You are a response validation assistant.

Question:
{question}

Retrieved Context:
{context}

Generated Answer:
{answer}

Instructions:

1. Check whether the answer is supported by the retrieved context.
2. If supported, return the answer unchanged.
3. If partially supported, improve it using only the retrieved context.
4. If unsupported, respond:

"I couldn't find enough information in the available documents to answer this accurately."

Return only the final validated answer.
"""
)

In [ ]:
validation_chain = validation_prompt | llm | StrOutputParser()

In [ ]:
def validate_response(state: CustomerSupportState):

    print("\n========== Response Validation Agent ==========")
    print("\n========== Validation Agent ==========")
    print("History:", len(state.get("conversation_history", [])))

    validated_answer = validation_chain.invoke(
        {
            "question": state["question"],
            "context": state["context"],
            "answer": state["answer"]
        }
    )

    print(f"Validated Answer:\n{validated_answer}")

    return {
        **state,
        "validated_answer": validated_answer,
        "validation_status": "validated"
    }

In [ ]:
# Register all nodes and build graph
builder = StateGraph(CustomerSupportState)

builder.add_node("initialize_agent", initialize_agent)
builder.add_node("classify_intent", classify_intent)
builder.add_node("product", product_workflow)
builder.add_node("warranty", warranty_workflow)
builder.add_node("shipping", shipping_workflow)
builder.add_node("returns", returns_workflow)
builder.add_node("care", care_workflow)
builder.add_node("recommendation", recommendation_workflow)
builder.add_node("general", general_workflow)
builder.add_node("unknown", unknown_workflow)
builder.add_node("retrieve_documents", retrieve_documents)
builder.add_node("generate_response", generate_response)
builder.add_node("rewrite_query", rewrite_query)
builder.add_node("validate_response", validate_response)

# Add edges
builder.add_edge(START, "initialize_agent")
builder.add_edge("initialize_agent", "classify_intent")
builder.add_edge("classify_intent", "rewrite_query")

builder.add_conditional_edges(
    "rewrite_query",
    route_intent,
    {
        ROUTE_PRODUCT: "product",
        ROUTE_WARRANTY: "warranty",
        ROUTE_RETURNS: "returns",
        ROUTE_SHIPPING: "shipping",
        ROUTE_CARE: "care",
        ROUTE_RECOMMENDATION: "recommendation",
        ROUTE_GENERAL: "general",
        ROUTE_UNKNOWN: "unknown",
    },
)

builder.add_edge("product", "retrieve_documents")
builder.add_edge("warranty", "retrieve_documents")
builder.add_edge("shipping", "retrieve_documents")
builder.add_edge("returns", "retrieve_documents")
builder.add_edge("care", "retrieve_documents")
builder.add_edge("recommendation", "retrieve_documents")
builder.add_edge("general", "retrieve_documents")
builder.add_edge("unknown", "retrieve_documents")

builder.add_edge("retrieve_documents", "generate_response")
builder.add_edge("generate_response", "validate_response")
builder.add_edge("validate_response",END)

customer_support_graph = builder.compile()

In [ ]:
### Validation
initial_state["question"] = "Can I insure my jewellery?"

In [ ]:
### Validation
result = customer_support_graph.invoke(initial_state)

In [ ]:
#### Updated class structure with recommendation
class CustomerSupportState(TypedDict):
    question: str

    rewritten_question: str

    intent: str

    documents: list

    context: str

    answer: str

    source_documents: list

    status: str

    validation_status: str

    validated_answer: str

    recommendations: list

    final_response: str

    conversation_history: List[Dict[str, str]]

    error: str | None

In [ ]:
### Recommendation Prompt
from langchain_core.prompts import ChatPromptTemplate

recommendation_prompt = ChatPromptTemplate.from_template(
"""
You are the Recommendation Agent for a jewellery customer support system.

Your responsibility is ONLY to recommend products.
Do NOT answer the customer's original question.
Do NOT repeat or modify the validated answer.

Customer Question:
{question}

Retrieved Catalogue:
{context}

Validated Answer:
{answer}

Instructions:

1. Read the validated answer carefully.

2. If the validated answer indicates that:
   - the requested product is unavailable,
   - out of stock,
   - not found,
   - or does not exist,

   then recommend 2-3 similar products ONLY from the retrieved catalogue.

3. For each recommendation include:
   - Product Name
   - Product ID (if available)
   - One short reason why it is a suitable alternative.

4. If the customer explicitly asks for recommendations or suggestions,
   recommend suitable products only from the retrieved catalogue.

5. Never invent products or product details.

6. Never recommend products that are not present in the retrieved context.

7. If no suitable alternatives exist in the retrieved catalogue,
   return exactly:

No Recommendation

8. Return ONLY the recommendation section.
Do NOT repeat the validated answer.
Do NOT include any introduction or conclusion.

Example Output:

• Sapphire Tennis Bracelet (ORN-S005)
  - Similar blue sapphire jewellery and currently in stock.

• Diamond Eternity Ring (ORN-D003)
  - Premium jewellery suitable as an alternative luxury purchase.
"""
)

In [ ]:
recommendation_chain = recommendation_prompt | llm | StrOutputParser()

In [ ]:
### Recommendation Agent

def recommendation_agent(state: CustomerSupportState):

    print("\n========== Recommendation Agent ==========")
    print("History:", len(state.get("conversation_history", [])))

    validated_answer = state["validated_answer"]

    # --------------------------------------------------
    # Skip recommendations when the answer is unavailable
    # --------------------------------------------------
    no_answer_phrases = [
        "couldn't find enough information",
        "not available in the provided documents",
        "don't have enough information",
        "cannot answer",
        "not found"
    ]

    if any(
        phrase in validated_answer.lower()
        for phrase in no_answer_phrases
    ):
        print("Skipping recommendations (information unavailable).")

        return {
            **state,
            "recommendations": "No Recommendation",
            "final_response": validated_answer,
            "status": "recommendation_skipped"
        }

    # --------------------------------------------------
    # Generate recommendations
    # --------------------------------------------------
    recommendations = recommendation_chain.invoke(
        {
            "question": state["question"],
            "context": state["context"],
            "answer": validated_answer
        }
    ).strip()

    print(recommendations)

    normalized = recommendations.lower().rstrip(".! ")

    if normalized == "no recommendation":
        final_response = validated_answer
    else:
        final_response = (
            validated_answer
            + "\n\nRecommended Products:\n"
            + recommendations
        )

    return {
        **state,
        "recommendations": recommendations,
        "final_response": final_response,
        "status": "recommendation_generated"
    }

In [ ]:
###Updating the final graph
#########################################################
########################################################
builder = StateGraph(CustomerSupportState)

# Register all nodes
builder.add_node("initialize_agent", initialize_agent)
builder.add_node("classify_intent", classify_intent)
builder.add_node("product", product_workflow)
builder.add_node("warranty", warranty_workflow)
builder.add_node("shipping", shipping_workflow)
builder.add_node("returns", returns_workflow)
builder.add_node("care", care_workflow)
builder.add_node("recommendation", recommendation_workflow)
builder.add_node("general", general_workflow)
builder.add_node("unknown", unknown_workflow)
builder.add_node("retrieve_documents", retrieve_documents)
builder.add_node("generate_response", generate_response)
builder.add_node("rewrite_query", rewrite_query)
builder.add_node("validate_response", validate_response)
builder.add_node("recommendation_agent", recommendation_agent)


# Add edges
builder.add_edge(START, "initialize_agent")
builder.add_edge("initialize_agent", "classify_intent")
builder.add_edge("classify_intent", "rewrite_query")

builder.add_conditional_edges(
    "rewrite_query",
    route_intent,
    {
        ROUTE_PRODUCT: "product",
        ROUTE_WARRANTY: "warranty",
        ROUTE_RETURNS: "returns",
        ROUTE_SHIPPING: "shipping",
        ROUTE_CARE: "care",
        ROUTE_RECOMMENDATION: "recommendation",
        ROUTE_GENERAL: "general",
        ROUTE_UNKNOWN: "unknown",
    },
)

builder.add_edge("product", "retrieve_documents")
builder.add_edge("warranty", "retrieve_documents")
builder.add_edge("shipping", "retrieve_documents")
builder.add_edge("returns", "retrieve_documents")
builder.add_edge("care", "retrieve_documents")
builder.add_edge("recommendation", "retrieve_documents")
builder.add_edge("general", "retrieve_documents")
builder.add_edge("unknown", "retrieve_documents")

builder.add_edge("retrieve_documents", "generate_response")
builder.add_edge("generate_response", "validate_response")
builder.add_edge("validate_response", "recommendation_agent")
builder.add_edge( "recommendation_agent",    END)

customer_support_graph = builder.compile()

In [ ]:
### This is used to recursive simulation. It will stay in the notebook
initial_state["conversation_history"] = []

In [ ]:
####Validation
initial_state["question"] = "Are there any bracelet in the catalogue"
result = customer_support_graph.invoke(initial_state)

In [ ]:
###Validation
initial_state = result        # Preserve updated state
initial_state["question"] = "What is the price of it?"

result = customer_support_graph.invoke(initial_state)

In [ ]:
###Validation
result = customer_support_graph.invoke(initial_state)

In [ ]:
####Validation 2
initial_state["question"] = "Tell me about the Sapphire Tennis Bracelet"

result = customer_support_graph.invoke(initial_state)

print(result["final_response"])

In [ ]:
###Validation 2
initial_state = result

In [ ]:
####Validation 2
initial_state["question"] = "What is the price of it?"

result = customer_support_graph.invoke(initial_state)

print(result["final_response"])

#Unit Testing

In [ ]:
initial_state["conversation_history"] = []

In [ ]:
initial_state["question"] = "Tell me about the Sapphire Tennis Bracelet"

result = customer_support_graph.invoke(initial_state)

print(result["final_response"])

In [ ]:
initial_state = result

In [ ]:
initial_state["question"] = "What is the price of it?"

result = customer_support_graph.invoke(initial_state)

print(result["final_response"])

#Streamlit Interface

#Testing